# Chatbot with tools

In [10]:
from dotenv import load_dotenv
import os

## Setup API Keys

In [11]:
load_dotenv()
SARVAM_API_KEY = os.getenv("SARVAM_API_KEY")

assert SARVAM_API_KEY is not None, "Could NOT load SARVAM_API_KEY from .env"

In [13]:
from sarvamai import SarvamAI
client = SarvamAI(
    api_subscription_key=SARVAM_API_KEY,
)

## Setting up Tools

In [25]:
def get_balance(account_number: str):
    if account_number == "001002":
        return "INR 50010"
    else:
        return "INR 400"
    
def get_transactions(account_number: str):
    if account_number == "001002":
        return [
            ("Time", "Transaction", "INR"),
            ("11:21 AM", "Debit - XYZ Super Market", "INR 651.50"),
            ("4:45 PM",  "Credit - Refund from PQR Braodband Services", "INR 999"),
        ]
    else:
        return "No transactions in the last 24 hrs in your account"

## Describing the Tools

In [26]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_balance",
            "description": "Get the balance for the account number",
            "parameters": {
                "type": "object",
                "properties": {
                    "account_number": {"type": "string", "description": "Account Number"},
                },
                "required": ["account_number"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_transactions",
            "description": "Get the transactions for the account number",
            "parameters": {
                "type": "object",
                "properties": {
                    "account_number": {"type": "string", "description": "Account Number"},
                },
                "required": ["account_number"],
            },
        },
    },
]

In [50]:
import json

In [78]:
def chat(messages: list):
    while True:
        response = client.chat.completions(
            model="sarvam-105b-conversations",
            messages=messages,
            tools=tools
        )

        if not response.choices[0].message.tool_calls:
            break        
    
        tool_calls = response.choices[0].message.tool_calls
        for tool_call in tool_calls:
            f_name = tool_call.function.name
            f_args = json.loads(tool_call.function.arguments)
    
            #Invoke the functions
            if(f_name == "get_balance"):
                result = get_balance(**f_args)
                print("******* ", result)
            elif(f_name == "get_transactions"):
                result = get_transactions(**f_args)
                print("******* ", result)
        
            messages.append(
                {
                    "role": "assistant",
                    "tool_calls": [
                        {
                            "id": tool_call.id,
                            "type": "function",
                            "function": {
                              "name": f_name,
                              "arguments": tool_call.function.arguments
                            }
                        }
                    ]
                }
            )
    
            messages.append(
                {
                    "role": "tool" ,
                    "content" : str(result),
                    "tool_call_id" : tool_call.id
                }
            )
    
        
    return response.choices[0].message.content

In [72]:
messages=[
    {"role": "system", "content": "You are a Customer Service Rep from ABC Bank."},
    {"role": "user", "content": "get my balance and transactions for 001002 and 1123"},
]

resp = chat(messages)
#print(resp.choices[0].message.content)
#print(resp.model_dump_json(indent = 2))

Here is the information for both your accounts:

**Account 001002**
- **Balance:** INR 50,010
- **Transactions (last 24 hours):**
  - 11:21 AM — Debit: INR 651.50 (XYZ Super Market)
  - 4:45 PM — Credit: INR 999.00 (Refund from PQR Broadband Services)

**Account 1123**
- **Balance:** INR 400
- **Transactions (last 24 hours):** No transactions recorded.

Is there anything else I can help you with today?


In [15]:
BLUE_TEXT = "\033[94m"
BLACK_TEXT = "\033[0m"

In [76]:
def run_chatbot():
    messages=[
        {"role": "system", "content": "You are a Customer Service Rep from ABC Bank."},
    ]
    
    while True:
        user_message = input("\nUser:")
        
        if user_message == "quit":
            break
            
        messages.append(
            {"role": "user", "content": user_message.strip()}
        )
    
        chat_response = chat(messages)
        print("\n", BLUE_TEXT, "Assistant: ", chat_response, BLACK_TEXT)
        messages.append(
            {"role": "assistant", "content": chat_response}
        )

In [79]:
run_chatbot()


User: Hi How are you ?



  Assistant:  Hello! I'm doing well, thank you for asking. How can I assist you with your account today? 



User: get my balance for 001002


*******  INR 50010

  Assistant:  Your current balance for account **001002** is **INR 50,010**. 

Is there anything else I can help you with today? 



User: transacetions


...


User: balance for 112233 and 001002



  Assistant:  Here are the balances for both accounts:

| Account Number | Balance       |
|----------------|---------------|
| **112233**     | INR 400       |
| **001002**     | INR 50,010    |

Is there anything else I can assist you with? 



User: balance for all accounts



  Assistant:  Here are the balances for all the accounts we've checked today:

| Account Number | Balance       |
|----------------|---------------|
| **001002**     | INR 50,010    |
| **112233**     | INR 400       |
| **12345b**     | INR 400       |

Is there anything else I can help you with? 



User: quit
